# Time Series Analysis & Forecasting — Nifty 50 Stock Market
## From Raw Prices to ARIMA Predictions

---
> **Dataset**: Nifty 50 Stock Market Data — 50 Indian large-cap stocks, daily OHLCV, Jan 2000 – Apr 2021  
> **Source**: [Rohan Rao @ Kaggle](https://www.kaggle.com/datasets/rohanrao/nifty50-stock-market-data) via KaggleHub

### What you will learn

| Step | Topic |
|------|-------|
| 1 | Download a Kaggle dataset programmatically with `kagglehub` |
| 2 | Understand the anatomy of stock market data (OHLCV) |
| 3 | Master `pandas` DatetimeIndex for time series operations |
| 4 | Visual EDA — trends, volume spikes, seasonality |
| 5 | Multi-stock comparison & correlation |
| 6 | Daily returns & volatility — the heart of financial analysis |
| 7 | Time resampling (daily → weekly → monthly) |
| 8 | Rolling windows & Bollinger Bands |
| 9 | Stationarity, differencing, ACF/PACF |
| 10 | ARIMA forecasting on monthly close prices |
| 11 | Forecast evaluation with error metrics |


---
## 1. Stock Market Time Series — Key Concepts

### 1.1 What Makes Stock Data Special?

Stock prices are one of the most studied time series in history, but they're also one of the **hardest to forecast** because:

1. **Random Walk Hypothesis** — prices already incorporate all known information (Efficient Market Hypothesis)
2. **Fat tails** — extreme moves (crashes, rallies) happen more often than a normal distribution predicts
3. **Regime changes** — a bull market and a bear market behave very differently
4. **Non-stationarity** — prices generally drift upward over decades (inflation, growth)

Despite these challenges, ARIMA models remain a **critical baseline** — if your fancy deep learning model can't beat ARIMA, it's probably not learning anything useful!

### 1.2 OHLCV — The Five Pillars of a Trading Day

```
  High ──────┐
             │  ← The candle body
  Open ──────┤  (filled if Close < Open = bearish day)
  Close ─────┤  (hollow if Close > Open = bullish day)
             │
  Low ───────┘

  Volume = total shares traded that day
  VWAP   = Volume Weighted Average Price
```

| Column | Full Name | What traders use it for |
|--------|-----------|-------------------------|
| **Open** | Opening price | Gap analysis, opening momentum |
| **High** | Day's high | Resistance levels |
| **Low** | Day's low | Support levels |
| **Close** | Closing price | Most used for analysis & modelling |
| **VWAP** | Volume Weighted Avg Price | Fair value benchmark for the day |
| **Volume** | Shares traded | Confirms trend strength |

### 1.3 What We Will Forecast

We will forecast the **monthly average closing price** of Reliance Industries (RELIANCE) — India's largest company.  
Monthly data smooths out daily noise and makes ARIMA patterns more interpretable.


---
## 2. Imports & Setup


In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Core data libraries ────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from datetime import datetime

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Time series statistics ─────────────────────────────────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats

# ── Kaggle data download ───────────────────────────────────────────────────
import kagglehub
import os

# ── Plot styling ───────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Colour palette — one colour per stock for consistent visuals throughout
STOCK_COLORS = {
    'RELIANCE': '#1f77b4',   # blue
    'TCS':      '#ff7f0e',   # orange
    'INFY':     '#2ca02c',   # green
    'HDFCBANK': '#d62728',   # red
    'MARUTI':   '#9467bd',   # purple
}

print('All libraries loaded!')

---
## 3. Download Data via KaggleHub

`kagglehub` lets you download any public Kaggle dataset with a single line of Python.  
On the first run it fetches and caches the dataset; subsequent runs use the local cache instantly.

```
kagglehub.dataset_download('owner/dataset-name')
    → returns the local path where files are stored
```


In [ ]:
# Download the Nifty50 dataset (cached after first run)
DATA_PATH = kagglehub.dataset_download('rohanrao/nifty50-stock-market-data')
print(f'Dataset downloaded to:\n  {DATA_PATH}\n')

# See what files are available
all_files = sorted(os.listdir(DATA_PATH))
stock_files = [f for f in all_files if f.endswith('.csv') and f != 'stock_metadata.csv' and 'NIFTY' not in f]
print(f'Total stock files : {len(stock_files)}')
print(f'Stocks available  : {[f.replace(".csv","") for f in stock_files]}')

In [ ]:
# ── Helper: load a single stock CSV ────────────────────────────────────────
def load_stock(ticker: str) -> pd.DataFrame:
    """
    Load stock CSV, parse dates, set DatetimeIndex.
    Returns a clean DataFrame with only EQ-series rows.
    """
    path = os.path.join(DATA_PATH, f'{ticker}.csv')
    df = pd.read_csv(path, parse_dates=['Date'])
    df = df[df['Series'] == 'EQ'].copy()   # keep only equity series (not futures/options)
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    df['Ticker'] = ticker
    return df

# Load the five stocks we'll use throughout this notebook
STOCKS = ['RELIANCE', 'TCS', 'INFY', 'HDFCBANK', 'MARUTI']
dfs = {ticker: load_stock(ticker) for ticker in STOCKS}

# Quick sanity check
for ticker, df in dfs.items():
    print(f'{ticker:12s} | {len(df):5d} trading days | {df.index[0].date()} → {df.index[-1].date()}')

# Main working frame = RELIANCE
rel = dfs['RELIANCE']

In [ ]:
# Show the first few rows of RELIANCE to understand the schema
cols_to_show = ['Open', 'High', 'Low', 'Close', 'VWAP', 'Volume', 'Deliverable Volume', '%Deliverble']
print('RELIANCE Industries — sample rows:')
rel[cols_to_show].head(8)

In [ ]:
# Dataset summary: shape, dtypes, missing values
print(f'Shape: {rel.shape}  ({rel.shape[0]} rows × {rel.shape[1]} columns)\n')
print('Column types & missing values:')
summary = pd.DataFrame({
    'dtype': rel.dtypes,
    'missing': rel.isnull().sum(),
    'missing %': (rel.isnull().mean() * 100).round(1)
})
print(summary)

---
## 4. Pandas DatetimeIndex — The Backbone of Time Series

When you set a `DatetimeIndex`, pandas unlocks powerful time-based operations:

```python
df['2015']               # all rows in 2015
df['2015-06':'2015-12']  # June to December 2015
df.resample('M').mean()  # collapse daily to monthly
df.rolling(30).mean()    # 30-day rolling average
df.shift(1)              # shift all values forward by 1 period
```

These 5 operations cover 80% of all time series data wrangling you'll ever do.


In [ ]:
# ── Extract calendar features from the DatetimeIndex ───────────────────────
# This turns implicit date information into explicit columns for EDA

rel = rel.copy()  # avoid SettingWithCopyWarning
rel['Year']       = rel.index.year
rel['Month']      = rel.index.month
rel['Month_name'] = rel.index.month_name()
rel['Quarter']    = rel.index.quarter
rel['Day_of_week']= rel.index.day_name()   # 'Monday', 'Tuesday', …
rel['Week']       = rel.index.isocalendar().week.astype(int)

print('Features added:')
rel[['Year','Month','Month_name','Quarter','Day_of_week']].head(6)

---
## 5. Visual Exploration — The Full Price History

Before touching a single model, **always plot the raw data** end-to-end.  
You are looking for:
- Overall upward/downward trend
- Obvious regime changes (crashes, rallies)
- Outlier spikes (often correspond to corporate events)
- Volume patterns (high volume = conviction behind the move)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1.5, 1.5]})

# ── Panel 1: Close price + 50-day and 200-day moving averages ─────────────
axes[0].plot(rel.index, rel['Close'], color='steelblue', linewidth=1.0, alpha=0.8, label='Close')
axes[0].plot(rel['Close'].rolling(50).mean(), color='orange', linewidth=1.8, label='50-day MA')
axes[0].plot(rel['Close'].rolling(200).mean(), color='darkred', linewidth=2.0, label='200-day MA')
axes[0].set_ylabel('Price (₹)', fontweight='bold')
axes[0].set_title('Reliance Industries — Full Price History (2000–2021)', fontweight='bold', fontsize=14)
axes[0].legend(loc='upper left')

# ── Panel 2: Daily trading range (High - Low) = intraday volatility ───────
rel['DailyRange'] = rel['High'] - rel['Low']
axes[1].bar(rel.index, rel['DailyRange'], color='steelblue', alpha=0.4, width=1)
axes[1].plot(rel['DailyRange'].rolling(50).mean(), color='navy', linewidth=1.5, label='50-day avg range')
axes[1].set_ylabel('Daily Range (₹)', fontweight='bold')
axes[1].legend(loc='upper left')

# ── Panel 3: Volume (shares traded per day) ────────────────────────────────
# Volume is in millions for readability
axes[2].bar(rel.index, rel['Volume'] / 1e6, color='seagreen', alpha=0.5, width=1)
axes[2].set_ylabel('Volume (M shares)', fontweight='bold')
axes[2].set_xlabel('Date')

# Annotate key market events
annotations = [
    ('2008-09-15', 'Lehman\nCrisis', 'red'),
    ('2020-03-23', 'COVID\nCrash',   'purple'),
    ('2016-11-08', 'Demonetisation', 'darkorange'),
]
for date_str, label, color in annotations:
    dt = pd.Timestamp(date_str)
    axes[0].axvline(dt, color=color, linestyle='--', linewidth=1.2, alpha=0.7)
    axes[0].text(dt, rel['Close'].max() * 0.95, label, color=color, fontsize=8,
                 ha='center', va='top', rotation=0,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()
print("""
Key observations:
  1. Strong long-term uptrend — consistent with India's economic growth.
  2. 2008 Lehman crash & 2020 COVID dip are clear — recovered strongly both times.
  3. Volume spikes often accompany price breakouts (high conviction moves).
  4. Daily range expanded massively in 2020 — market uncertainty = wider daily swings.
""")

---
## 6. Exploratory Data Analysis — Slicing Time

### 6.1 Year-over-Year Price Trend


In [ ]:
# Annual statistics: mean close price and total volume per year
annual = rel.groupby('Year').agg(
    Mean_Close=('Close', 'mean'),
    Max_Close=('Close', 'max'),
    Min_Close=('Close', 'min'),
    Total_Volume=('Volume', 'sum')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Candlestick-style annual range (min/max) with mean bar
axes[0].bar(annual['Year'], annual['Max_Close'] - annual['Min_Close'],
            bottom=annual['Min_Close'], color='lightsteelblue', edgecolor='steelblue',
            alpha=0.7, width=0.7, label='Price Range (Min–Max)')
axes[0].plot(annual['Year'], annual['Mean_Close'], 'D-',
             color='navy', markersize=6, linewidth=1.8, label='Mean Close')
axes[0].set_title('Annual Close Price Range — RELIANCE', fontweight='bold')
axes[0].set_ylabel('Price (₹)')
axes[0].set_xlabel('Year')
axes[0].legend()

# Annual trading volume
axes[1].bar(annual['Year'], annual['Total_Volume'] / 1e9, color='seagreen', alpha=0.8, width=0.7)
axes[1].set_title('Annual Trading Volume — RELIANCE', fontweight='bold')
axes[1].set_ylabel('Total Volume (Billion shares)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

### 6.2 Monthly & Day-of-Week Seasonality

Stock markets are **not expected to have strong monthly seasonality** (unlike rainfall),  
but there are known effects like the "January Effect" or "Monday Effect" — let's check!


In [ ]:
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
dow_order   = ['Monday','Tuesday','Wednesday','Thursday','Friday']

# Daily return = percentage change from previous close
rel['Daily_Return'] = rel['Close'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Monthly average return (not VWAP level, but returns — more comparable across years)
sns.boxplot(data=rel.dropna(subset=['Daily_Return']),
            x='Month_name', y='Daily_Return',
            order=month_order, palette='Blues', ax=axes[0])
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Daily Returns by Month — RELIANCE', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Daily Return (%)')
axes[0].tick_params(axis='x', rotation=45)

# Day-of-week effect
sns.boxplot(data=rel.dropna(subset=['Daily_Return']),
            x='Day_of_week', y='Daily_Return',
            order=dow_order, palette='Greens', ax=axes[1])
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Daily Returns by Day of Week — RELIANCE', fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Daily Return (%)')

plt.tight_layout()
plt.show()
print("""
The median return (orange line in each box) stays close to zero across all months and days.
This is consistent with the Efficient Market Hypothesis — no reliable pattern to exploit.
The FAT TAILS (outliers as dots) tell the real story: extreme moves can happen any day.
""")

---
## 7. Daily Returns & Volatility Analysis

### Why return instead of price?

```
Price  : 100 → 200 → 150   (absolute, non-stationary, hard to compare)
Return :  +100%  → -25%    (percentage change, stationary, comparable across stocks)
```

**Volatility** = how much returns vary = standard deviation of returns.  
High volatility = risky stock. Low volatility = stable stock.


In [ ]:
# Compute daily returns for all 5 stocks
returns = pd.DataFrame({
    ticker: dfs[ticker]['Close'].pct_change() * 100
    for ticker in STOCKS
})

# ── Return distribution: histogram + stats ─────────────────────────────────
fig, axes = plt.subplots(1, len(STOCKS), figsize=(20, 4), sharey=True)
for ax, ticker in zip(axes, STOCKS):
    data = returns[ticker].dropna()
    ax.hist(data, bins=80, color=STOCK_COLORS[ticker], alpha=0.7, edgecolor='white', density=True)
    # Overlay a normal distribution for comparison
    x = np.linspace(data.min(), data.max(), 300)
    ax.plot(x, stats.norm.pdf(x, data.mean(), data.std()), 'k--', linewidth=1.5, label='Normal')
    ax.set_title(f'{ticker}\n(σ={data.std():.2f}%)', fontweight='bold')
    ax.set_xlabel('Daily Return (%)')
    ax.axvline(0, color='red', linewidth=1)

axes[0].set_ylabel('Density')
axes[0].legend()
plt.suptitle('Daily Return Distributions — Fat Tails vs Normal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("""
The black dashed line is what a normal distribution would look like.
Notice the actual bars are TALLER in the centre and have LONGER tails → 'leptokurtic' or 'fat-tailed'.
This means crashes/rallies happen MORE OFTEN than normal distribution would predict.
""")

In [ ]:
# Return statistics summary
stats_df = returns.agg(['mean', 'std', 'min', 'max',
                         lambda x: x.kurt(),
                         lambda x: x.skew()]).T
stats_df.columns = ['Mean Return %', 'Std Dev %', 'Worst Day %', 'Best Day %', 'Kurtosis', 'Skewness']
stats_df = stats_df.round(3)

# Kurtosis > 3 (or excess kurtosis > 0 in scipy) = fat tails
print("Daily Return Statistics (entire history):")
print(stats_df.to_string())
print("\nNote: Kurtosis > 3 confirms fat tails in all stocks (crashes/rallies more likely than normal)")

---
## 8. Multi-Stock Comparison & Correlation

### 8.1 Normalised Price — Who Grew the Most?

To compare stocks on a common scale, we **normalise** prices to 100 at the start:  
`normalised_t = (price_t / price_start) × 100`


In [ ]:
# Align all stocks to a common start date
close_prices = pd.DataFrame({
    ticker: dfs[ticker]['Close']
    for ticker in STOCKS
})

# Use 2004 onwards; TCS IPO was Aug 2004 so early rows are NaN for TCS.
# ffill() propagates prices forward for any odd missing trading days.
close_prices = close_prices.loc['2004':].dropna(how='all')
close_prices.ffill(inplace=True)

# Normalise each stock to 100 at its OWN first valid date.
# bfill().iloc[0] gets the first non-NaN row even when row 0 has NaN for some tickers
# (avoids division by NaN which would make the entire TCS column NaN).
first_valid = close_prices.bfill().iloc[0]
normalised  = close_prices.div(first_valid) * 100

fig, ax = plt.subplots(figsize=(16, 6))
for ticker in STOCKS:
    series = normalised[ticker].dropna()
    if series.empty:
        continue
    ax.plot(series.index, series.values,
            color=STOCK_COLORS[ticker], linewidth=1.5, label=ticker)
    # Label the final multiplier (e.g. 2000x means 20x growth from base=100)
    final_val = series.iloc[-1]
    ax.annotate(f'{ticker}\n{final_val:.0f}x',
                xy=(series.index[-1], final_val),
                xytext=(10, 0), textcoords='offset points',
                fontsize=8, color=STOCK_COLORS[ticker], fontweight='bold')

ax.axhline(100, color='black', linestyle='--', linewidth=0.8, alpha=0.5, label='Baseline = 100')
ax.set_title('Normalised Price Growth (Base = 100 at first trading date)', fontweight='bold', fontsize=14)
ax.set_ylabel('Normalised Price (Base = 100)')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 8.2 Correlation Heatmap — Do They Move Together?


In [ ]:
# Correlation on RETURNS (not prices) — price correlation is spurious due to trend
ret_corr = returns[STOCKS].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price correlation (misleading — shows spurious high correlation due to shared trend)
price_corr = close_prices[STOCKS].corr()
sns.heatmap(price_corr, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=axes[0], square=True)
axes[0].set_title('Price Correlation\n(⚠ misleading — shared trend inflates correlation)', fontweight='bold')

# Returns correlation (correct way)
sns.heatmap(ret_corr, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=axes[1], square=True)
axes[1].set_title('Daily Return Correlation\n(✓ correct — removes trend effect)', fontweight='bold')

plt.tight_layout()
plt.show()
print("""
Key insight: Always compute correlation on RETURNS, not raw prices!
All Nifty50 stocks are positively correlated (they all react to market-wide news),
but IT stocks (TCS, INFY) correlate more with each other than with RELIANCE or HDFCBANK.
""")

---
## 9. Time Resampling — From Daily Noise to Monthly Signal

`resample()` is like `groupby()` but for time — it buckets rows by a time period.

| Offset Code | Meaning |
|-------------|---------|
| `'D'` | Daily (no change) |
| `'W'` | Weekly (last trading day of each week) |
| `'MS'` | Monthly Start |
| `'ME'` | Monthly End |
| `'QS'` | Quarterly Start |
| `'YS'` | Yearly Start |


In [ ]:
rel_close = rel['Close']

# Resample to different frequencies
rel_weekly    = rel_close.resample('W').mean()     # average price during the week
rel_monthly   = rel_close.resample('ME').mean()    # average price during the month
rel_quarterly = rel_close.resample('QE').mean()    # average price during the quarter

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Focus on 2015-2021 for visual clarity
ZOOM = slice('2015', None)
pairs = [
    (rel_close[ZOOM],       'Daily',     'steelblue',  axes[0, 0]),
    (rel_weekly[ZOOM],      'Weekly',    'orange',     axes[0, 1]),
    (rel_monthly[ZOOM],     'Monthly',   'seagreen',   axes[1, 0]),
    (rel_quarterly[ZOOM],   'Quarterly', 'tomato',     axes[1, 1]),
]

for series, label, color, ax in pairs:
    ax.plot(series.index, series.values, color=color, linewidth=1.5, marker='.' if label != 'Daily' else '', markersize=4)
    ax.set_title(f'{label} Average Close Price (2015–2021)', fontweight='bold')
    ax.set_ylabel('Price (₹)')
    ax.set_xlabel('Date')

plt.suptitle('Resampling: Same Data at Different Time Granularities', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("""
Notice how coarser resampling smooths out noise:
  Daily   → very noisy, hard to see the trend
  Weekly  → smoother, short-term swings visible
  Monthly → clear trend, good for ARIMA modelling
  Quarterly → broad strokes, good for long-term analysis

For ARIMA forecasting we'll use MONTHLY data — enough observations, clean signal.
""")

---
## 10. Rolling Windows — Moving Averages & Bollinger Bands

A **rolling window** slides across the series and computes a statistic over the last N observations.  
It's used to:
- Smooth noisy prices (moving average)
- Measure changing volatility (rolling std)
- Build technical indicators like Bollinger Bands

### Bollinger Bands
```
Upper Band = 20-day MA + 2 × 20-day rolling std
Middle Band = 20-day MA
Lower Band = 20-day MA - 2 × 20-day rolling std

When price hits the upper band → potentially overbought
When price hits the lower band → potentially oversold
```


In [ ]:
# Use 2019-2021 window for Bollinger Bands (interesting COVID period)
zoom = rel_close['2019':'2021']

window = 20   # 20 trading days ≈ 1 calendar month

ma20  = zoom.rolling(window).mean()
std20 = zoom.rolling(window).std()

upper_band = ma20 + 2 * std20
lower_band = ma20 - 2 * std20

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

# ── Price + Bollinger Bands ────────────────────────────────────────────────
axes[0].plot(zoom.index, zoom.values, color='steelblue', linewidth=1.2, label='Close Price', alpha=0.9)
axes[0].plot(ma20, color='orange', linewidth=2.0, label='20-day MA (middle band)')
axes[0].plot(upper_band, color='red', linewidth=1.2, linestyle='--', label='Upper Band (+2σ)')
axes[0].plot(lower_band, color='green', linewidth=1.2, linestyle='--', label='Lower Band (−2σ)')
axes[0].fill_between(zoom.index, lower_band, upper_band, alpha=0.08, color='gray')
axes[0].set_title('RELIANCE — Bollinger Bands (20-day, 2σ), 2019–2021', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Price (₹)')
axes[0].legend(loc='upper left', fontsize=9)

# ── Rolling volatility (bandwidth = upper - lower) ─────────────────────────
bandwidth = upper_band - lower_band
axes[1].fill_between(bandwidth.index, bandwidth.values, color='gray', alpha=0.4)
axes[1].plot(bandwidth, color='black', linewidth=1.2)
axes[1].set_title('Band Width (Upper − Lower) = Volatility Proxy', fontweight='bold')
axes[1].set_ylabel('Band Width (₹)')
axes[1].set_xlabel('Date')

# Mark COVID crash
for ax in axes:
    ax.axvline(pd.Timestamp('2020-03-23'), color='purple', linestyle=':', linewidth=1.5)

axes[0].text(pd.Timestamp('2020-03-23'), zoom.max() * 0.98, 'COVID\nLow',
             color='purple', fontsize=8, ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Rolling volatility comparison across stocks ─────────────────────────────
# 30-day rolling annualised volatility (multiply by sqrt(252) for annual)

fig, ax = plt.subplots(figsize=(16, 5))
for ticker in STOCKS:
    daily_ret = dfs[ticker]['Close'].pct_change()
    roll_vol  = daily_ret.rolling(30).std() * np.sqrt(252) * 100   # annualised %
    ax.plot(roll_vol.loc['2015':], color=STOCK_COLORS[ticker], linewidth=1.3, label=ticker, alpha=0.85)

ax.axvline(pd.Timestamp('2020-03-23'), color='black', linestyle=':', linewidth=1.2)
ax.text(pd.Timestamp('2020-03-23'), ax.get_ylim()[1] * 0.95, ' COVID', fontsize=9, color='black')

ax.set_title('30-day Rolling Annualised Volatility (%) — 2015–2021', fontweight='bold', fontsize=13)
ax.set_ylabel('Volatility (% annualised)')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.show()
print("COVID spike in March 2020 is visible across ALL stocks — a systemic shock, not company-specific.")

---
## 11. Time Shifting — Lead/Lag Analysis

`shift(n)` moves the series **n periods forward** (positive n) or backward (negative n).  

Financial uses:
- `shift(1)` → previous day's price (used to compute daily return: `close / close.shift(1) - 1`)
- `shift(-1)` → tomorrow's price (used to create prediction target in ML models)
- Cross-correlation: does INFY lead TCS by a day? Check `corr(TCS, INFY.shift(1))`


In [ ]:
zoom2 = rel_close['2020-01':'2020-06']

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, n, title in [
    (axes[0], -5,  'Shifted BACKWARD 5 days\n("peek at future" — don\'t use in real models!)'),
    (axes[1],  0,  'Original series'),
    (axes[2], +5,  'Shifted FORWARD 5 days\n("lagged" — commonly used in features)'),
]:
    ax.plot(zoom2.index, zoom2.values, color='gray', alpha=0.4, label='Original')
    shifted = zoom2.shift(n)
    ax.plot(shifted.index, shifted.values, color='steelblue', linewidth=2, label=f'shift({n})')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Date')
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=30)

axes[0].set_ylabel('Price (₹)')
plt.suptitle('Time Shifting — RELIANCE Close Price (Jan–Jun 2020)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Lead-lag correlation between INFY and TCS
print("\nLead-lag cross-correlations (INFY returns vs TCS returns):")
infy_ret = dfs['INFY']['Close'].pct_change().dropna()
tcs_ret  = dfs['TCS']['Close'].pct_change().dropna()
common   = infy_ret.align(tcs_ret, join='inner')
for lag in [-2, -1, 0, 1, 2]:
    corr = common[0].corr(common[1].shift(lag))
    direction = 'TCS leads INFY' if lag < 0 else ('same day' if lag == 0 else 'INFY leads TCS')
    print(f'  Lag {lag:+d} ({direction:20s}): correlation = {corr:.4f}')

---
## 12. Preparing for ARIMA — Stationarity Tests

We will model the **monthly average close price** of RELIANCE.  
First, let's check if it's stationary (it almost certainly isn't — stock prices drift upward).


In [ ]:
# Monthly average close price of RELIANCE — this is our modelling target
monthly_close = rel_close.resample('ME').mean()
monthly_close.name = 'RELIANCE_Close'

print(f'Monthly series: {len(monthly_close)} months  ({monthly_close.index[0].date()} → {monthly_close.index[-1].date()})')

# ADF test helper (reused from Notebook 03)
def adf_report(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    stat, pval, lags, nobs, crit, _ = result
    verdict = 'STATIONARY ✓' if pval < 0.05 else 'NON-STATIONARY ✗'
    print(f"""
  ── ADF Test: {name} ──────────────────────────────────
  ADF Statistic  : {stat:>10.4f}
  p-value        : {pval:>10.4f}   {'< 0.05 → reject H₀' if pval < 0.05 else '>= 0.05 → fail to reject H₀'}
  Critical (5%)  : {crit['5%']:>10.4f}
  Verdict        : {verdict}
""")
    return pval < 0.05

# Check the raw series first
print("\n1) Raw monthly close prices:")
adf_report(monthly_close, 'Monthly Close')

# First-order differencing (removes linear trend)
diff1 = monthly_close.diff(1)
print("2) After 1st difference (month-over-month change):")
adf_report(diff1, '1st Diff of Monthly Close')

# Log transform helps stabilise variance before differencing
log_monthly = np.log(monthly_close)
log_diff1   = log_monthly.diff(1)
print("3) Log-price, then 1st difference (= log-returns):")
adf_report(log_diff1, 'Log-Diff Monthly Close')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=False)

for ax, series, title, color in [
    (axes[0], monthly_close, 'Monthly Close Price (Non-Stationary: upward trend)', 'steelblue'),
    (axes[1], diff1,         '1st Difference: Month-over-Month Change in Price (₹)', 'darkorange'),
    (axes[2], log_diff1,     'Log-Differenced Series ≈ Monthly Log-Returns (Stationary)', 'seagreen'),
]:
    ax.plot(series.dropna(), color=color, linewidth=1.2)
    ax.axhline(series.mean(), color='black', linestyle='--', linewidth=1.0,
               label=f'mean = {series.mean():.2f}')
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

---
## 13. ACF & PACF — Identifying p, d, q

We apply ACF and PACF on the **stationary** differenced log-price series.

```
Recall the reading guide:
  PACF cuts off at lag p  → AR(p) model
  ACF  cuts off at lag q  → MA(q) model
  Both tail off slowly    → ARMA(p, q) model
```


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 9))

plot_acf(monthly_close.dropna(),  lags=36, alpha=0.05, ax=axes[0, 0],
         title='ACF — Raw Monthly Close (non-stationary)')
plot_pacf(monthly_close.dropna(), lags=36, alpha=0.05, ax=axes[0, 1],
          title='PACF — Raw Monthly Close', method='ywm')

plot_acf(log_diff1.dropna(),  lags=36, alpha=0.05, ax=axes[1, 0],
         title='ACF — Log-Differenced (stationary) ← use this!')
plot_pacf(log_diff1.dropna(), lags=36, alpha=0.05, ax=axes[1, 1],
          title='PACF — Log-Differenced ← use this!', method='ywm')

for ax in axes.flatten():
    ax.set_xlabel('Lag (months)')

plt.tight_layout()
plt.show()
print("""
Interpretation of the stationary ACF/PACF (bottom row):
  Most lags fall within the confidence band (shaded region).
  This is close to a white noise process — consistent with the Random Walk Hypothesis.
  We'll try ARIMA(1,1,1) as a standard baseline for stock prices.
  The 'd=1' because we need one regular differencing to achieve stationarity.
""")

---
## 14. Box-Jenkins 3-Step Methodology — Applied to Stock Prices

```
Step 1: IDENTIFICATION
  • ADF confirms d=1 (one regular difference needed)
  • ACF/PACF of differenced series suggest low p, q (both ≤ 2)
  • No clear seasonal pattern (stock markets don't have monsoons!)
  • We test: ARIMA(0,1,1), ARIMA(1,1,0), ARIMA(1,1,1), ARIMA(2,1,1)

Step 2: ESTIMATION
  • Fit each model with Maximum Likelihood
  • Compare AIC (lower is better)

Step 3: DIAGNOSTIC
  • Residuals ACF: no significant spikes
  • Ljung-Box p > 0.05
```


In [ ]:
# Hold out last 12 months (May 2020 – Apr 2021) for evaluation
CUTOFF = '2020-04'
train_monthly = monthly_close[:CUTOFF]
test_monthly  = monthly_close[pd.Timestamp(CUTOFF) + pd.DateOffset(months=1):]

print(f'Training : {len(train_monthly)} months  → {train_monthly.index[-1].strftime("%b %Y")}')
print(f'Test     : {len(test_monthly)}  months  → {test_monthly.index[0].strftime("%b %Y")} to {test_monthly.index[-1].strftime("%b %Y")}')

# Compare candidate ARIMA orders by AIC
candidate_orders = [(0,1,1), (1,1,0), (1,1,1), (2,1,1), (1,1,2), (2,1,2)]
results = []

print('\nComparing candidate ARIMA models on training data:')
print(f'  {"Order":15s}  {"AIC":>10s}  {"BIC":>10s}')
print('  ' + '-' * 38)

for order in candidate_orders:
    try:
        m = SARIMAX(train_monthly, order=order,
                    enforce_stationarity=False, enforce_invertibility=False)
        fitted_m = m.fit(disp=False)
        results.append({'order': order, 'AIC': fitted_m.aic, 'BIC': fitted_m.bic})
        print(f'  ARIMA{str(order):12s}  {fitted_m.aic:>10.2f}  {fitted_m.bic:>10.2f}')
    except Exception as e:
        print(f'  ARIMA{str(order):12s}  FAILED: {e}')

best_row   = min(results, key=lambda x: x['AIC'])
best_order = best_row['order']
print(f'\n→ Best by AIC: ARIMA{best_order}  (AIC = {best_row["AIC"]:.2f})')

---
## 15. Model Estimation — Fit the Best ARIMA


In [ ]:
print(f'Fitting ARIMA{best_order} on RELIANCE monthly close prices...\n')

final_model = SARIMAX(
    train_monthly,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
final_fit = final_model.fit(disp=False)
print(final_fit.summary())

---
## 16. Diagnostic Checks — Is the Model Adequate?


In [ ]:
residuals = final_fit.resid

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── 1. Residual time series ────────────────────────────────────────────────
axes[0, 0].plot(residuals, color='gray', linewidth=0.9)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuals over Time\n(should look like random noise around zero)', fontweight='bold')
axes[0, 0].set_ylabel('Residual (₹)')

# ── 2. Histogram of residuals ──────────────────────────────────────────────
axes[0, 1].hist(residuals, bins=40, color='steelblue', edgecolor='white', density=True)
x_range = np.linspace(residuals.min(), residuals.max(), 200)
axes[0, 1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
                'r--', linewidth=2, label='Normal PDF')
axes[0, 1].set_title('Histogram of Residuals\n(should resemble a bell curve)', fontweight='bold')
axes[0, 1].legend()

# ── 3. ACF of residuals ────────────────────────────────────────────────────
plot_acf(residuals.dropna(), lags=24, alpha=0.05, ax=axes[1, 0],
         title='ACF of Residuals\n(all bars should be inside the shaded band)')
axes[1, 0].set_xlabel('Lag (months)')

# ── 4. Q-Q plot ────────────────────────────────────────────────────────────
(osm, osr), (slope, intercept, r) = stats.probplot(residuals.dropna())
axes[1, 1].scatter(osm, osr, s=8, color='tomato', alpha=0.6)
axes[1, 1].plot(osm, slope * np.array(osm) + intercept, 'k-', linewidth=1.5)
axes[1, 1].set_title(f'Q-Q Plot  (R² = {r**2:.3f})\n(points near the line → near-normal residuals)', fontweight='bold')
axes[1, 1].set_xlabel('Theoretical Quantiles')
axes[1, 1].set_ylabel('Sample Quantiles')

plt.suptitle(f'Diagnostic Plots — ARIMA{best_order}  RELIANCE Monthly Close', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box formal test for residual independence
lb = acorr_ljungbox(residuals.dropna(), lags=[6, 12, 18, 24], return_df=True)
print('Ljung-Box Test for Residual Autocorrelation:')
print('H0: residuals are white noise  |  H1: autocorrelation present')
print()
print(lb.to_string())
print()
if (lb['lb_pvalue'] > 0.05).all():
    print('PASS: All p-values > 0.05 => Residuals are WHITE NOISE => Model is adequate!')
else:
    failed_lags = lb[lb['lb_pvalue'] <= 0.05].index.tolist()
    print(f'FAIL: Significant autocorrelation at lags: {failed_lags} => consider increasing p or q')

---
## 17. Forecasting the Next 12 Months

We use the trained model to forecast **12 months** beyond the training window.  
We also compare against the held-out test set (May 2020 – Apr 2021) to evaluate accuracy.

> **Important caveat**: Stock price forecasting is genuinely hard.  
> ARIMA captures the average behaviour of the historical series.  
> It cannot predict black swans (COVID crash), earnings surprises, or policy changes.


In [ ]:
n_forecast = len(test_monthly)

forecast_result = final_fit.get_forecast(steps=n_forecast)
forecast_mean   = forecast_result.predicted_mean
forecast_ci     = forecast_result.conf_int(alpha=0.05)

# Build the summary comparison table
forecast_df = pd.DataFrame({
    'Month':     [d.strftime('%b %Y') for d in forecast_mean.index],
    'Actual':    test_monthly.values[:n_forecast].round(2),
    'Forecast':  forecast_mean.values.round(2),
    'Lower 95%': forecast_ci.iloc[:, 0].values.round(2),
    'Upper 95%': forecast_ci.iloc[:, 1].values.round(2),
})
forecast_df['Error'] = (forecast_df['Actual'] - forecast_df['Forecast']).round(2)

# Use plain ASCII markers to avoid encoding issues across platforms
forecast_df['In CI?'] = forecast_df.apply(
    lambda r: 'YES' if r['Lower 95%'] <= r['Actual'] <= r['Upper 95%'] else 'NO', axis=1
)

print(f'ARIMA{best_order} — 12-Month Forecast vs Actual for RELIANCE\n')
print(forecast_df.to_string(index=False))

In [ ]:
# ── Main forecast visualisation ────────────────────────────────────────────
plot_from = '2018'

fig, ax = plt.subplots(figsize=(16, 6))

# Full training history from 2018 onward
ax.plot(train_monthly[plot_from:], color='steelblue', linewidth=1.8, label='Historical (train)')

# Actual test values
ax.plot(test_monthly, color='seagreen', linewidth=2.5, marker='o', markersize=6,
        label='Actual (held-out test)', zorder=5)

# Forecast
ax.plot(forecast_mean, color='tomato', linewidth=2.5, linestyle='--', marker='s', markersize=6,
        label=f'ARIMA{best_order} Forecast', zorder=5)

# Confidence interval
ax.fill_between(
    forecast_mean.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    color='tomato', alpha=0.12, label='95% Confidence Interval'
)

# Vertical line at forecast start
ax.axvline(train_monthly.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast start')

# COVID crash marker
ax.axvline(pd.Timestamp('2020-03-23'), color='purple', linestyle='-.', linewidth=1.2, alpha=0.7)
ax.text(pd.Timestamp('2020-03-23'), ax.get_ylim()[0] * 1.05, 'COVID\nCrash',
        color='purple', fontsize=8, ha='center')

ax.set_title(f'RELIANCE Monthly Close — ARIMA{best_order} Forecast vs Actual', fontweight='bold', fontsize=14)
ax.set_ylabel('Average Monthly Close Price (₹)')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

---
## 18. Forecast Evaluation


In [ ]:
actual_arr   = test_monthly.values[:n_forecast]
forecast_arr = forecast_mean.values

mae  = mean_absolute_error(actual_arr, forecast_arr)
rmse = np.sqrt(mean_squared_error(actual_arr, forecast_arr))
mape = np.mean(np.abs((actual_arr - forecast_arr) / actual_arr)) * 100

# Count how many actuals fell inside the 95% confidence interval
in_ci_count = (forecast_df['In CI?'] == 'YES').sum()
in_ci_pct   = in_ci_count / n_forecast * 100

print('Forecast Evaluation — RELIANCE Monthly Close Price\n')
print(f'  MAE        = INR {mae:>8.2f}   (average absolute error per month)')
print(f'  RMSE       = INR {rmse:>8.2f}   (root mean squared error — penalises large misses)')
print(f'  MAPE       =  {mape:>7.2f}%   (average % error — scale-independent)')
print(f'  In CI      =  {in_ci_pct:>6.1f}%   ({in_ci_count}/{n_forecast} actuals within 95% CI)')
print()
quality = 'acceptable for a simple baseline' if mape < 20 else 'high — expected for volatile stocks'
print(f'  MAPE of {mape:.0f}% is {quality}.')
print(f'  {in_ci_pct:.0f}% of actuals are captured inside the 95% CI.')
print('  A well-calibrated model should capture ~95% of actuals in its CI.')

In [ ]:
# ── Actual vs Forecast scatter + error bar chart ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: perfect forecast would lie on y=x
axes[0].scatter(actual_arr, forecast_arr, color='steelblue', s=80, zorder=5, alpha=0.8)
lim = (min(actual_arr.min(), forecast_arr.min()) * 0.95,
       max(actual_arr.max(), forecast_arr.max()) * 1.05)
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect forecast (y = x)')
axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_title('Actual vs Forecast (Monthly Close)', fontweight='bold')
axes[0].set_xlabel('Actual (₹)'); axes[0].set_ylabel('Forecast (₹)')
axes[0].legend()

# Monthly error bar chart
errors = actual_arr - forecast_arr
colors = ['seagreen' if e >= 0 else 'tomato' for e in errors]
axes[1].bar(range(len(errors)), errors, color=colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(range(len(forecast_df)))
axes[1].set_xticklabels(forecast_df['Month'], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Forecast Error per Month (green=over-estimated, red=under-estimated)', fontweight='bold')
axes[1].set_ylabel('Error = Actual − Forecast (₹)')

plt.tight_layout()
plt.show()

---
## 19. Bonus — Future Forecast Beyond Known Data

Train on all available data (2000–2021) and forecast the **next 12 months into the unknown**.


In [ ]:
# Refit on ALL data
full_model = SARIMAX(
    monthly_close,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

future_fc    = full_model.get_forecast(steps=12)
future_mean  = future_fc.predicted_mean
future_ci    = future_fc.conf_int(alpha=0.05)

future_df = pd.DataFrame({
    'Month':     [d.strftime('%b %Y') for d in future_mean.index],
    'Forecast':  future_mean.values.round(2),
    'Lower 95%': future_ci.iloc[:, 0].values.round(2),
    'Upper 95%': future_ci.iloc[:, 1].values.round(2),
})

print(f'RELIANCE — 12-Month Ahead Forecast (trained on full 2000–2021 history)\n')
print(future_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# Last 3 years of history
ax.plot(monthly_close['2018':], color='steelblue', linewidth=2.0, label='Historical Close')

# Future forecast
ax.plot(future_mean, color='tomato', linewidth=2.5, linestyle='--', marker='D', markersize=7,
        label='12-Month Forecast')
ax.fill_between(future_mean.index, future_ci.iloc[:, 0], future_ci.iloc[:, 1],
                color='tomato', alpha=0.15, label='95% Confidence Interval')

# Annotate each forecast point
for date, val in zip(future_mean.index, future_mean.values):
    ax.annotate(f'₹{val:.0f}', xy=(date, val),
                xytext=(0, 14), textcoords='offset points',
                ha='center', fontsize=8, color='darkred', fontweight='bold')

ax.axvline(monthly_close.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast start')

ax.set_title(f'RELIANCE — ARIMA{best_order} Forecast: Next 12 Months', fontweight='bold', fontsize=14)
ax.set_ylabel('Average Monthly Close Price (₹)')
ax.set_xlabel('Date')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print("""
Notice the WIDE confidence interval — this is ARIMA being honest about uncertainty.
The further out you forecast, the wider the band grows.
A narrow CI would be suspicious; stock prices are genuinely uncertain!
""")

---
## 20. Summary, Key Takeaways & What's Next

### What We Built — In One Picture

```
Kaggle Dataset (50 stocks × 21 years × daily OHLCV)
     │
     ├─ EDA: price history, volume, events (crash, COVID)
     ├─ Returns: % change → comparable across stocks
     ├─ Volatility: rolling std → changes over time (ARCH effects)
     ├─ Correlation: return correlation (not price!)
     ├─ Resampling: daily → monthly (for ARIMA)
     ├─ Rolling windows: MA, Bollinger Bands
     ├─ Stationarity: ADF test → d=1 needed
     ├─ ACF/PACF → p, q identification
     ├─ ARIMA fit → parameter estimation (MLE)
     ├─ Diagnostics: Ljung-Box, ACF of residuals
     └─ Forecast: 12 months with 95% CI
```

### Key Takeaways for ML/DS Students

| Lesson | Detail |
|--------|--------|
| **Use returns, not prices** | Returns are stationary; raw prices are not. Correlation on prices is misleading. |
| **Always test stationarity** | ADF test before modelling. ARIMA needs d differences to become stationary. |
| **ACF/PACF identify p, q** | But for stock data, both are often near zero — markets are efficient! |
| **Confidence intervals matter** | Wide CI = honest uncertainty. Don't use point forecasts alone. |
| **ARIMA is a baseline** | Beat it before claiming your ML model works. |
| **Residual diagnostics** | A model is only complete after checking residuals are white noise. |

### Limitations of ARIMA for Stock Prices

1. **Assumes linear relationships** — markets have complex nonlinear dynamics
2. **Cannot use external information** — earnings, news, macro events → use ARIMAX or LSTM
3. **Constant variance assumed** — volatility clusters (use GARCH for volatility modelling)
4. **Short-term only** — reliability drops sharply beyond 3–6 months

### What's Next?

| Model | What it adds over ARIMA |
|-------|-------------------------|
| **ARIMAX / SARIMAX** | External regressors (e.g., Nifty50 index as input) |
| **GARCH** | Time-varying volatility modelling |
| **Prophet (Meta)** | Handles multiple seasonalities, holidays automatically |
| **LSTM / GRU** | Deep learning — can learn complex nonlinear patterns |
| **Transformer (TFT)** | State-of-the-art for multi-step time series forecasting |
